# 번역 Scaled Dot-Product Attention 모델 작성
- " I love you" => "난 너를 사랑해"


In [1]:
import numpy as np
import math

src_tokens = ['I', 'love', 'you']
tgt_tokens = ['난', '너를', '사랑해']

# 쿼리,키,벡터 초기화
n_src = len(src_tokens)
n_tgt = len(tgt_tokens)
K = np.eye(n_src)
V = np.eye(n_src)
Q = np.zeros((n_tgt, n_src))

# 출력 단어(tgt_tokens) 하나하나에 그 단어가 어느 입력 단어(src_tokens)에 집중할 지를 설정
for i in range(n_tgt):

  if i == 0: # 출력 첫 단어(i==0)는
    Q[i,0] = 1.0  # 입력 첫단어(src[0])에 100% 집중

  elif i == n_tgt -1 :# 출력 마지막 단어(i==n_tgt -1)는
    Q[i,-1] = 1.0 # 입력 마지막단어(src[0])에 100% 집중

  else: # 중간 단어는
    Q[i,i:i + 2] = 0.5 # 분산해서 집중

print(Q)

[[1.  0.  0. ]
 [0.  0.5 0.5]
 [0.  0.  1. ]]


In [3]:
def scaled_atten_func(q, K, V):
  # 유사도 점수 계산(내적) + 스케일을 위한 정규화 - 논문에 있는 수식 √dt
  scores = q.dot(K.T) / math.sqrt(K.shape[1])
  # softmax로 가중치 구하기
  exp = np.exp(scores - np.max(scores))
  weights = exp / exp.sum() # 가중치 합(Attention Weight)

  # Attention Weight * V 합 : 동적인 Context Vector - seq2seq는 없음.
  # weights[:, None] * V : weights[:, None]를 V와 브로드캐스팅을 하기 위해 차원을 맞춰주는것뿐임
  context = (weights[:, None] * V).sum(axis=0)
  print("context :", context)
  return context, weights


# 각 목표 단어 마다 Attention실행 후 결과 확인
generated = []  # 생성된 단어 저장용
for i, target in enumerate(tgt_tokens):
  context, weights = scaled_atten_func(Q[i], K, V)
  print(f'생성 단어 : {target}')
  print("입력 단어 별 Attention Weight :")
  for src_tokens, w in zip(src_tokens, weights):
    print(f'{src_tokens:>7}:{w:.3f}')
    generated.append(target)
    print()

print('최종 번역 결과: ')
print(' '.join(generated))

context : [0.47108308 0.26445846 0.26445846]
생성 단어 : 난
입력 단어 별 Attention Weight :
      I:0.471

   love:0.264

    you:0.264

context : [0.27253035 0.36373483 0.36373483]
생성 단어 : 너를
입력 단어 별 Attention Weight :
      y:0.273

      o:0.364

      u:0.364

context : [0.26445846 0.26445846 0.47108308]
생성 단어 : 사랑해
입력 단어 별 Attention Weight :
      u:0.264

최종 번역 결과: 
난 난 난 너를 너를 너를 사랑해
